In [ ]:
!pip install -q langchain-community faiss-cpu langchain-huggingface sentence-transformers

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

more_documents = [
    Document(page_content=(
        "Cosine similarity measures how close two vectors are by looking at the angle between them. "
        "A score near 1 means the texts have very similar meaning, and a score near 0 means they "
        "are unrelated."
    )),
    Document(page_content=(
        "Maximal Marginal Relevance, or MMR, picks results that are relevant to the query but "
        "different from each other. It reduces repeated answers when several chunks say almost "
        "the same thing."
    )),
    Document(page_content=(
        "Hybrid search combines keyword search such as BM25 with semantic vector search. "
        "Keywords catch exact names and terms, while vectors catch similar meaning, so together "
        "they find more of the right documents."
    )),
    Document(page_content=(
        "The context window is the maximum amount of text a language model can read at once. "
        "In RAG, only the best retrieved chunks are placed in the prompt so they fit inside "
        "this limit."
    )),
    Document(page_content=(
        "Metadata filtering narrows a search to documents with specific properties, such as a "
        "source, a date or a category. The filter is applied first, and then the remaining "
        "documents are ranked by similarity."
    )),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(more_documents, embeddings)

/tmp/ipykernel_1039/3923140122.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5},
)

def show(retriever, query):
    print("Query:", query)
    for i, d in enumerate(retriever.invoke(query), 1):
        print(f"  {i}. {d.page_content[:70]}...")

query = "How can I avoid getting repeated results in search?"

print("SIMILARITY")
show(similarity_retriever, query)
print("\nMMR")
show(mmr_retriever, query)

SIMILARITY
Query: How can I avoid getting repeated results in search?
  1. Maximal Marginal Relevance, or MMR, picks results that are relevant to...
  2. Hybrid search combines keyword search such as BM25 with semantic vecto...
  3. Metadata filtering narrows a search to documents with specific propert...

MMR
Query: How can I avoid getting repeated results in search?
  1. Maximal Marginal Relevance, or MMR, picks results that are relevant to...
  2. Metadata filtering narrows a search to documents with specific propert...
  3. Hybrid search combines keyword search such as BM25 with semantic vecto...


In [ ]:
results = vector_store.max_marginal_relevance_search(
    query,
    k=3,
    fetch_k=5,
    lambda_mult=0.5,
)
for d in results:
    print("-", d.page_content[:70])

- Maximal Marginal Relevance, or MMR, picks results that are relevant to
- Metadata filtering narrows a search to documents with specific propert
- Hybrid search combines keyword search such as BM25 with semantic vecto


In [ ]:
duplicates = [
    Document(page_content=(
        "Cosine similarity compares two vectors using the angle between them. "
        "A value close to 1 means the texts are very similar in meaning."
    )),
    Document(page_content=(
        "To measure how similar two embeddings are, cosine similarity looks at the angle "
        "between the vectors. Scores near 1 mean a very close meaning."
    )),
]

dup_store = FAISS.from_documents(more_documents + duplicates, embeddings)

q = "How is the similarity between two vectors measured?"

print("SIMILARITY")
show(dup_store.as_retriever(search_type="similarity", search_kwargs={"k": 3}), q)

print("\nMMR")
show(dup_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 7, "lambda_mult": 0.3},
), q)

SIMILARITY
Query: How is the similarity between two vectors measured?
  1. To measure how similar two embeddings are, cosine similarity looks at ...
  2. Cosine similarity measures how close two vectors are by looking at the...
  3. Cosine similarity compares two vectors using the angle between them. A...

MMR
Query: How is the similarity between two vectors measured?
  1. To measure how similar two embeddings are, cosine similarity looks at ...
  2. Metadata filtering narrows a search to documents with specific propert...
  3. The context window is the maximum amount of text a language model can ...
